In [10]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [11]:
import pandas as pd
from datetime import datetime

# Define time period
start = "2025-04-23T00:00:00Z"  # adjust as needed

# List of owners to pull from
list_of_owners = ['HTOC Org', 'HTOC Comm', 'HTOC legacy data']
final_results = []
group_id = 5629499544001057

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        # Build TQL query string
        tql = f'ownerName EQ "{owner}" and lastModified >= "{start}"'

        # Set up the request
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/groups/{group_id}?tql={tql}&fields=associatedGroups,associatedIndicators&resultStart=0&resultLimit=10000')

        # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Ensure 'data' is a list for each result
    normalized_results = []
    for result in final_results:
        if isinstance(result.get('data'), dict):
            normalized_results.append(result['data'])
        elif isinstance(result.get('data'), list):
            normalized_results.extend(result['data'])

    if normalized_results:
        observed_src = pd.json_normalize(normalized_results)
        print(f"\nRetrieved {len(observed_src)} observed indicators.")
    else:
        print("\nNo valid data to normalize.")
else:
    print("\nNo indicators retrieved.")



Querying owner: HTOC Org

Querying owner: HTOC Comm

Querying owner: HTOC legacy data

Retrieved 3 observed indicators.


In [12]:
# Unnest 'associatedIndicators.data' and extract all 'summary' values into a flat list
indicator_summaries = []

for indicators in observed_src['associatedIndicators.data']:
    if isinstance(indicators, list):
        for indicator in indicators:
            summary = indicator.get('summary')
            if summary:
                indicator_summaries.append(summary)

# Create a DataFrame from the extracted indicator summaries
indicator_df = pd.DataFrame({'summary': indicator_summaries})
indicator_df = indicator_df.drop_duplicates()
display(indicator_df)

,summary
0,206.189.57.182
1,104.248.203.234
2,171.241.78.71
3,159.65.128.194
4,147.28.240.218
...,...
1129,45.90.185.100
1130,64.94.85.248
1131,45.90.185.107
1132,91.228.126.9


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Enrich final filtered indicators (Shodan & VirusTotalV3) and flatten results
# ─────────────────────────────────────────────────────────────────────────────
from concurrent.futures import ThreadPoolExecutor
from itertools import chain
import urllib.parse as ul
import pandas as pd
import logging

logger = logging.getLogger(__name__)
COL_PATH = "data.enrichment.data"        # adjust if the API changes
MAX_WORKERS = 32                         # tune for your env / rate-limit

# --------------------------------------------------------------------------- #
# 1. Build the list of indicator values
# --------------------------------------------------------------------------- #
indicator_values = (
    indicator_df["summary"]
    .astype("string")     # nullable, faster than object
    .dropna()
    .unique()
    .tolist()
)
logger.info("Enriching %d indicators with Shodan & VirusTotalV3 …",
            len(indicator_values))

# --------------------------------------------------------------------------- #
# 2. One helper to enrich a single indicator (to be run in parallel)
# --------------------------------------------------------------------------- #
def enrich_one(value: str) -> dict:
    enc = ul.quote_plus(value, safe="")                      # encode path
    q = ul.urlencode({"type": ["Shodan", "VirusTotalV3"]},
                     doseq=True)                             # ?type=x&type=y
    ro = RequestObject()
    ro.set_http_method("POST")
    ro.set_request_uri(f"/v3/indicators/{enc}/enrich?{q}")
    ro.set_body({})

    resp = tc.api_request(ro)
    resp.raise_for_status()

    payload = resp.json()
    payload["summary"] = value
    return payload

# --------------------------------------------------------------------------- #
# 3. Fire all requests concurrently
# --------------------------------------------------------------------------- #
enriched, failed = [], []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    for val, res in zip(indicator_values, pool.map(enrich_one, indicator_values)):
        try:
            enriched.append(res)
        except Exception as e:
            failed.append({"summary": val, "error": str(e)})

# --------------------------------------------------------------------------- #
# 4. Merge enrichment payload back into the base DataFrame
# --------------------------------------------------------------------------- #
if not enriched:
    logger.warning("No enrichment data retrieved.")
else:
    df_enriched = (
        pd.json_normalize(enriched)
          .drop_duplicates("summary", keep="last")
    )
    indicator_df = indicator_df.merge(df_enriched, on="summary", how="left")

    # ----------------------------------------------------------------------- #
    # 5. Flatten the nested enrichment column (if present)
    # ----------------------------------------------------------------------- #
    if COL_PATH in indicator_df:
        # explode so each row has exactly ONE dict instead of a list
        tmp = (
            indicator_df.loc[indicator_df[COL_PATH].notna(), ["summary", COL_PATH]]
                       .explode(COL_PATH)
        )
        # keep only dicts (skip stray scalars / None)
        tmp = tmp[tmp[COL_PATH].apply(lambda x: isinstance(x, dict))]

        enrich_flat = (
            pd.json_normalize(tmp[COL_PATH], errors="ignore")
              .add_prefix("enrich_")
        )
        enrich_flat["summary"] = tmp["summary"].values

        # ----- custom aggregation helpers ---------------------------------- #
        def _agg(series: pd.Series):
            flat = list(chain.from_iterable(
                v if isinstance(v, list) else [v] for v in series.dropna()
            ))
            if not flat:
                return None
            if all(not isinstance(v, (list, dict)) for v in flat):
                return pd.unique(flat).tolist()
            return flat   # keep nested structures intact

        obj_cols = enrich_flat.select_dtypes("object").columns.difference(["summary"])
        num_cols = enrich_flat.columns.difference(obj_cols.union(["summary"]))
        agg_map  = {**{c: _agg for c in obj_cols}, **{c: "max" for c in num_cols}}

        wide = enrich_flat.groupby("summary", as_index=False).agg(agg_map)

        # merge the flattened columns back in and drop raw list column
        indicator_df = (
            indicator_df.drop(columns=[COL_PATH])
                        .drop_duplicates("summary")
                        .merge(wide, on="summary", how="left")
        )

        logger.info("Successfully enriched & merged %d indicators.", len(wide))
    else:
        logger.info("Enrichment column %s not found; nothing to flatten.", COL_PATH)

# --------------------------------------------------------------------------- #
# 6. Optional: report failures
# --------------------------------------------------------------------------- #
if failed:
    logger.warning("%d indicators failed to enrich.", len(failed))



C:\Users\jaskew\AppData\Local\Temp\ipykernel_1272\2619612472.py:95: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  return pd.unique(flat).tolist()


In [14]:
df_enriched

,status,summary,data.id,data.dateAdded,data.ownerId,data.ownerName,data.webLink,data.type,data.lastModified,data.rating,data.confidence,data.description,data.summary,data.privateFlag,data.active,data.activeLocked,data.ip,data.legacyLink,data.enrichment.data
0,Success,206.189.57.182,6755399460007564,2025-07-02T12:05:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-25T15:18:20Z,4.0,71,"Details\nIn late June 2025, Iran-aligned hackt...",206.189.57.182,False,True,False,206.189.57.182,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'tags': ['cloud'], 'cloudP..."
1,Success,104.248.203.234,6755399460007434,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-25T10:12:55Z,4.0,71,"Details\nIn late June 2025, Iran-aligned hackt...",104.248.203.234,False,True,False,104.248.203.234,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'tags': ['cloud'], 'cloudP..."
2,Success,171.241.78.71,6755399460008056,2025-07-02T12:05:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-25T09:13:24Z,4.0,71,"Details\nIn late June 2025, Iran-aligned hackt...",171.241.78.71,False,True,False,171.241.78.71,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'hostNames': ['dynamic-ip-..."
3,Success,159.65.128.194,6755399460008047,2025-07-02T12:05:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-25T09:12:44Z,4.0,71,"Details\nIn late June 2025, Iran-aligned hackt...",159.65.128.194,False,True,False,159.65.128.194,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'tags': ['cloud'], 'cloudP..."
4,Success,147.28.240.218,6755399460008029,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-25T09:12:43Z,4.0,71,"Details\nIn late June 2025, Iran-aligned hackt...",147.28.240.218,False,True,False,147.28.240.218,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'hostNames': ['gateway.zsc..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1129,Success,45.90.185.100,5629499555060687,2025-06-16T17:42:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-18T13:37:58Z,3.0,78,HTOC-20250703-1249,45.90.185.100,False,True,False,45.90.185.100,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 7}]"
1130,Success,64.94.85.248,5272874,2025-01-31T11:46:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-18T13:37:58Z,3.0,78,HTOC-20250703-1249,64.94.85.248,False,True,False,64.94.85.248,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 9}]"
1131,Success,45.90.185.107,6755399459033764,2025-06-16T17:42:46Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-18T13:37:57Z,3.0,78,HTOC-20250703-1249,45.90.185.107,False,True,False,45.90.185.107,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 7}]"
1132,Success,91.228.126.9,6755399460008515,2025-07-02T12:05:38Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-18T13:37:56Z,4.0,71,"Details\nIn late June 2025, Iran-aligned hackt...",91.228.126.9,False,True,False,91.228.126.9,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 0}]"


In [15]:
# Aggregate recent_tags by 'summary', combining lists and taking max for numeric columns
def flatten_and_dedupe(series):
    vals = [v for v in series.dropna()]
    out = []
    for v in vals:
        if isinstance(v, list):
            out.extend(v)
        else:
            out.append(v)
    # Dedupe if all are hashable and not dict/list
    if all(not isinstance(v, (list, dict)) for v in out):
        return list(pd.unique(out))
    return out

agg_dict = {col: flatten_and_dedupe for col in obj_cols}
agg_dict.update({col: "max" for col in num_cols})

# Use 'wide' DataFrame, which contains the enrichment columns
recent_tags_agg = wide[["summary", "enrich_vtMaliciousCount"]].copy()
display(recent_tags_agg)


,summary,enrich_vtMaliciousCount
0,1.4.195.14,2.0
1,1.53.108.237,0.0
2,1.54.101.176,0.0
3,10.243.1.11,0.0
4,101.32.254.55,0.0
...,...,...
1129,96.44.137.177,0.0
1130,96.9.125.211,8.0
1131,97.78.58.180,0.0
1132,98.152.200.61,6.0


In [16]:
# Display percentages of all unique values in enrich_vtMaliciousCount
value_counts = recent_tags_agg['enrich_vtMaliciousCount'].value_counts(normalize=True) * 100
print(value_counts.sort_index().round(2))


enrich_vtMaliciousCount
0.0     47.62
1.0     25.22
2.0     12.08
3.0      4.59
4.0      1.85
5.0      1.68
6.0      1.50
7.0      1.76
8.0      1.32
9.0      1.06
10.0     0.09
12.0     0.18
13.0     0.79
14.0     0.09
15.0     0.18
Name: proportion, dtype: float64
